In [11]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
from pathlib import Path
import subprocess

REPO_DIR = Path(
    "/content/Low-Light-Image-Enhancement-and-Downstream-Recognition"
)

if not REPO_DIR.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth", "1",
            "https://github.com/NMH1203/Low-Light-Image-Enhancement-and-Downstream-Recognition.git",
            str(REPO_DIR),
        ],
        check=True,
    )

SOURCE_DATASET = (
    REPO_DIR
    / "Dataset"
    / "exdark_yolo_dark"
)

OUTPUT_DATASET = Path(
    "/content/drive/MyDrive/Project18_Datasets/exdark_yolo_clahe_only"
)

assert SOURCE_DATASET.exists(), (
    f"Không tìm thấy dataset nguồn: {SOURCE_DATASET}"
)

print("Dataset nguồn:", SOURCE_DATASET)
print("Dataset kết quả:", OUTPUT_DATASET)

Dataset nguồn: /content/Low-Light-Image-Enhancement-and-Downstream-Recognition/Dataset/exdark_yolo_dark
Dataset kết quả: /content/drive/MyDrive/Project18_Datasets/exdark_yolo_clahe_only


In [13]:
import cv2
import numpy as np


def enhance_clahe_only(
    image_bgr: np.ndarray,
) -> np.ndarray:
    """
    CLAHE thuần trên kênh sáng L của không gian màu LAB.
    Không sử dụng Bilateral Filter.
    """

    if image_bgr is None or image_bgr.size == 0:
        raise ValueError("Input image is empty.")

    # Chuyển BGR sang LAB
    lab = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2LAB,
    )

    # Tách kênh L, A và B
    luminance, channel_a, channel_b = cv2.split(lab)

    # Chỉ áp dụng CLAHE trên kênh sáng L
    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8),
    )

    luminance_clahe = clahe.apply(
        luminance
    )

    # Ghép lại các kênh LAB
    enhanced_lab = cv2.merge(
        (
            luminance_clahe,
            channel_a,
            channel_b,
        )
    )

    # Chuyển LAB về BGR
    enhanced_bgr = cv2.cvtColor(
        enhanced_lab,
        cv2.COLOR_LAB2BGR,
    )

    # Trả kết quả ngay, không qua Bilateral Filter
    return enhanced_bgr

In [14]:
import os
import cv2
import shutil

from tqdm.auto import tqdm

SPLITS = [
    "train",
    "valid",
    "test",
]

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
    ".tif",
    ".tiff",
}

processed_count = 0
skipped_count = 0

for split in SPLITS:
    source_images_dir = (
        SOURCE_DATASET
        / split
        / "images"
    )

    source_labels_dir = (
        SOURCE_DATASET
        / split
        / "labels"
    )

    output_images_dir = (
        OUTPUT_DATASET
        / split
        / "images"
    )

    output_labels_dir = (
        OUTPUT_DATASET
        / split
        / "labels"
    )

    if not source_images_dir.exists():
        raise FileNotFoundError(
            f"Thiếu thư mục ảnh: {source_images_dir}"
        )

    if not source_labels_dir.exists():
        raise FileNotFoundError(
            f"Thiếu thư mục label: {source_labels_dir}"
        )

    output_images_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    output_labels_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    image_paths = sorted(
        path
        for path in source_images_dir.iterdir()
        if (
            path.is_file()
            and path.suffix.lower() in IMAGE_EXTENSIONS
        )
    )

    label_paths = sorted(
        source_labels_dir.glob("*.txt")
    )

    print(
        f"\n{split}: "
        f"{len(image_paths)} ảnh, "
        f"{len(label_paths)} labels"
    )

    if len(image_paths) != len(label_paths):
        raise ValueError(
            f"Số ảnh và labels không bằng nhau trong {split}"
        )

    # CLAHE không thay đổi hình học nên sao chép nguyên labels
    for label_path in label_paths:
        destination_label = (
            output_labels_dir
            / label_path.name
        )

        shutil.copy2(
            label_path,
            destination_label,
        )

    # Xử lý tuần tự, không dùng workers
    for image_path in tqdm(
        image_paths,
        desc=f"CLAHE-only {split}",
    ):
        output_path = (
            output_images_dir
            / f"{image_path.stem}.png"
        )

        # Tiếp tục được nếu phiên Colab bị ngắt
        if (
            output_path.exists()
            and output_path.stat().st_size > 0
        ):
            skipped_count += 1
            continue

        image_bgr = cv2.imread(
            str(image_path),
            cv2.IMREAD_COLOR,
        )

        if image_bgr is None:
            raise ValueError(
                f"Không đọc được ảnh: {image_path}"
            )

        original_shape = image_bgr.shape

        enhanced_bgr = enhance_clahe_only(
            image_bgr
        )

        if enhanced_bgr.shape != original_shape:
            raise ValueError(
                f"Kích thước ảnh bị thay đổi: {image_path}"
            )

        # Lưu tạm trước để tránh ảnh lỗi khi Colab bị ngắt
        temporary_path = output_path.with_name(
            output_path.stem + ".temporary.png"
        )

        success = cv2.imwrite(
            str(temporary_path),
            enhanced_bgr,
            [cv2.IMWRITE_PNG_COMPRESSION, 3],
        )

        if not success:
            raise IOError(
                f"Không lưu được ảnh: {output_path}"
            )

        os.replace(
            temporary_path,
            output_path,
        )

        processed_count += 1

print("\nĐã xử lý ảnh.")
print("Ảnh mới:", processed_count)
print("Ảnh bỏ qua vì đã tồn tại:", skipped_count)


train: 5142 ảnh, 5142 labels


CLAHE-only train:   0%|          | 0/5142 [00:00<?, ?it/s]


valid: 1469 ảnh, 1469 labels


CLAHE-only valid:   0%|          | 0/1469 [00:00<?, ?it/s]


test: 734 ảnh, 734 labels


CLAHE-only test:   0%|          | 0/734 [00:00<?, ?it/s]


Đã xử lý ảnh.
Ảnh mới: 0
Ảnh bỏ qua vì đã tồn tại: 7345


In [15]:
# =========================================================
# Tạo data.yaml
# =========================================================
source_yaml = (
    SOURCE_DATASET
    / "data.yaml"
)

output_yaml = (
    OUTPUT_DATASET
    / "data.yaml"
)

with open(
    source_yaml,
    "r",
    encoding="utf-8",
) as file:
    config = yaml.safe_load(file)

config["path"] = OUTPUT_DATASET.as_posix()
config["train"] = "train/images"
config["val"] = "valid/images"
config["test"] = "test/images"

if "names" in config:
    config["nc"] = len(config["names"])

with open(
    output_yaml,
    "w",
    encoding="utf-8",
) as file:
    yaml.safe_dump(
        config,
        file,
        sort_keys=False,
        allow_unicode=True,
    )

In [16]:
# =========================================================
# Kiểm tra kết quả
# =========================================================
print("\n===== KIỂM TRA DATASET =====")

for split in SPLITS:
    image_count = len(
        list(
            (
                OUTPUT_DATASET
                / split
                / "images"
            ).glob("*.png")
        )
    )

    label_count = len(
        list(
            (
                OUTPUT_DATASET
                / split
                / "labels"
            ).glob("*.txt")
        )
    )

    status = (
        "OK"
        if image_count == label_count
        else "LỖI"
    )

    print(
        f"{split:5s}: "
        f"{image_count} ảnh, "
        f"{label_count} labels — "
        f"{status}"
    )

    assert image_count == label_count


print("\n===== HOÀN THÀNH =====")
print("Ảnh mới xử lý:", processed_count)
print("Ảnh đã tồn tại:", skipped_count)
print("Dataset:", OUTPUT_DATASET)
print("data.yaml:", output_yaml)


===== KIỂM TRA DATASET =====
train: 5142 ảnh, 5142 labels — OK
valid: 1469 ảnh, 1469 labels — OK
test : 734 ảnh, 734 labels — OK

===== HOÀN THÀNH =====
Ảnh mới xử lý: 0
Ảnh đã tồn tại: 7345
Dataset: /content/drive/MyDrive/Project18_Datasets/exdark_yolo_clahe_only
data.yaml: /content/drive/MyDrive/Project18_Datasets/exdark_yolo_clahe_only/data.yaml
